# TP : Mise en œuvre d'un Pipeline ETL et d'un Data Lakehouse (Bronze / Silver / Gold)
### Projet E-shop — Data Engineering avec Apache Spark & PySpark
**Auteur :** ilyassTakmout
**Date :** Novembre 2026
**Objectif :** Concevoir et implémenter un pipeline de données de bout en bout selon l'architecture médaillon (Bronze, Silver, Gold), assurant la qualité des données, la gestion des anomalies, l'idempotence, la persistance et la production d'indicateurs commerciaux conformes aux résultats de référence.


---
## Étape 1 : Préparation de l'environnement et ingestion

L'objectif de cette étape est d'initialiser l'environnement d'exécution Apache Spark, de charger les 5 sources de données brutes (`customers`, `products`, `orders`, `order_items`, `promotions`), de définir les schémas d'ingestion et les schémas cibles, et de caractériser les anomalies structurelles présentes dans les données sources.


### 1. Organisation de l'arborescence du projet E-shop
L'architecture de répertoires a été mise en place conformément aux recommandations du sujet :
```
E-shop/
├── data/
│   └── source/                  <- Fichiers bruts d'origine
│       ├── customers.csv
│       ├── products.csv
│       ├── orders.csv
│       ├── order_items.csv
│       └── promotions.jsonl
├── data_lakehouse/
│   ├── bronze/                  <- Données brutes ingérées & Parquet
│   ├── silver/                  <- Données nettoyées, typées et validées
│   ├── gold/                    <- Vues métier et agrégations analytiques
│   ├── quarantine/              <- Enregistrements rejetés avec motifs
│   └── metadata/                <- Manifestes d'ingestion et rapports d'audit
├── notebooks/
│   └── TP_E-shop.ipynb          <- Notebook interactif complet
└── scripts/
    └── etl_pipeline.py          <- Pipeline ETL automatisé et idempotent
```


### 2. Initialisation de la session Apache Spark


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)
from pyspark.sql.window import Window
import os
import json
import shutil
import uuid
from datetime import datetime

# Initialisation d'une session Spark avec support UTC et politique de parser corrigée
spark = (
    SparkSession.builder
    .appName("E-shop Data Lakehouse")
    .master("local[*]")
    .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


### 3. Affichage des métadonnées de la session Spark


In [ ]:
print("=== SPARK SESSION METADATA ===")
print("Version de Spark       :", spark.version)
print("Master utilisé         :", spark.sparkContext.master)
print("Nom de l'application   :", spark.sparkContext.appName)
print("Identifiant application:", spark.sparkContext.applicationId)


### 4. Lecture des cinq sources dans des DataFrames
Conformément aux bonnes pratiques du sujet, les fichiers CSV sont initialement lus sans inférence automatique de schéma (`inferSchema=False`), préservant tous les champs en `StringType` afin de capturer fidèlement les valeurs d'origine et de détecter les erreurs de format.


In [ ]:
# Chemins relatifs adaptés au conteneur et à l'exécution locale
BASE_SOURCE_PATH = "data/source"

customers_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(f"{BASE_SOURCE_PATH}/customers.csv")
)

products_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(f"{BASE_SOURCE_PATH}/products.csv")
)

orders_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(f"{BASE_SOURCE_PATH}/orders.csv")
)

order_items_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(f"{BASE_SOURCE_PATH}/order_items.csv")
)

# Lecture des promotions en format texte brut pour un parsing JSON contrôlé
promotions_raw_text = spark.read.text(f"{BASE_SOURCE_PATH}/promotions.jsonl")


### 5. Schémas d'ingestion (Bronze) vs Schémas métier cibles (Silver) et justifications

#### Justification des types retenus :
1. **Identifiants (`customer_id`, `product_id`, `order_id`, `order_item_id`) :** `StringType`. Même s'ils contiennent parfois des chiffres, les identifiants sont des clés alphanumériques (ex. `C001`, `P005`). Les traiter en numérique entraînerait la perte des zéros non significatifs et interdirait les préfixes textuels.
2. **Dates (`registration_date`, `order_date`, `start_date`, `end_date`) :** `DateType` (format standard ISO `yyyy-MM-dd`). Permet les calculs d'intervalles temporels, le tri chronologique et l'extraction de composantes (mois, année).
3. **Quantités (`quantity`) :** `IntegerType`. Une quantité commandée est un nombre entier discret. Elle doit être strictement positive ($> 0$).
4. **Prix et coûts (`unit_price`, `unit_cost`) :** `DoubleType` (ou `DecimalType(10,2)`). Représente des valeurs monétaires continues nécessitant des calculs précis de marge et de chiffre d'affaires.
5. **Taux de remise (`discount_rate`) :** `DoubleType`. Représente un pourcentage décimal compris dans l'intervalle $[0.0, 1.0]$.


In [ ]:
# 1. Schémas d'ingestion Bronze (tout en StringType)
schema_bronze_customers = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("region", StringType(), True),
    StructField("registration_date", StringType(), True)
])

schema_bronze_products = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("unit_cost", StringType(), True)
])

schema_bronze_orders = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("promotion_code", StringType(), True)
])

schema_bronze_order_items = StructType([
    StructField("order_item_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True)
])

# 2. Schémas métier Silver (Types enrichis et castés)
schema_silver_customers = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), False),
    StructField("region", StringType(), False),
    StructField("registration_date", DateType(), False)
])

schema_silver_products = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("unit_cost", DoubleType(), False)
])

schema_silver_orders = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_date", DateType(), False),
    StructField("status", StringType(), False),
    StructField("promotion_code", StringType(), True)
])

schema_silver_order_items = StructType([
    StructField("order_item_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False)
])


### 6. Affichage des schémas et aperçu des premières lignes de chaque source


In [ ]:
sources_raw = {
    "customers": customers_raw,
    "products": products_raw,
    "orders": orders_raw,
    "order_items": order_items_raw,
    "promotions (texte brut)": promotions_raw_text
}

for name, df in sources_raw.items():
    print(f"\n================ {name.upper()} ================")
    df.printSchema()
    df.show(5, truncate=False)


### 7. Contrôle des volumes et comparaison avec l'Annexe A


In [ ]:
expected_volumes = {
    "customers": 32,
    "products": 14,
    "orders": 44,
    "order_items": 86,
    "promotions (texte brut)": 6
}

print(f"{'Source':<25} | {'Volume Lu':<10} | {'Attendu':<10} | {'Conformité'}")
print("-" * 60)
for name, df in sources_raw.items():
    cnt = df.count()
    exp = expected_volumes[name]
    status = "CONFORME" if cnt == exp else "ANOMALIE DE VOLUME"
    print(f"{name:<25} | {cnt:<10} | {exp:<10} | {status}")


### 8. Identification des anomalies et difficultés dans les données brutes


In [ ]:
print("--- 1. Dates invalides (ex. '2025-02-30' dans customers) ---")
customers_raw.filter(F.col("registration_date") == "2025-02-30").show()

print("--- 2. Valeurs nulles ou vides (ex. C031 sans nom) ---")
customers_raw.filter(
    F.col("customer_name").isNull() | (F.trim(F.col("customer_name")) == "")
).show()

print("--- 3. Quantités non numériques ou aberrantes dans order_items ---")
order_items_raw.filter(
    ~F.col("quantity").rlike("^[0-9]+$") | (F.col("quantity") == "0")
).show()

print("--- 4. Coûts négatifs dans products (ex. P013) ---")
products_raw.filter(F.col("unit_cost").cast("double") < 0).show()

print("--- 5. Ligne JSON mal formée dans promotions ---")
promotions_raw_text.filter(~F.col("value").endswith("}")).show(truncate=False)


### Questions d'analyse — Étape 1

#### 1. Pourquoi est-il utile de contrôler les schémas avant les transformations ?
Le contrôle des schémas en amont permet d'appliquer le principe de **Fail-Fast** :
- Il protège les jobs Spark contre les pannes tardives en cours de traitement causées par des types incompatibles.
- Il garantit l'alignement entre le contrat d'interface des données sources (schéma attendu) et leur réalité physique.
- Il empêche la propagation silencieuse de valeurs erronées ou corrompues dans les couches analytiques aval.

#### 2. Quelle différence existe-t-il entre une lecture réussie et une ingestion de qualité ?
- **Lecture réussie :** Opération purement technique où le moteur Spark parvient à désérialiser les octets du fichier en lignes et colonnes, sans lever d'erreur I/O. Elle n'offre aucune garantie sur la validité du contenu (un fichier CSV plein de valeurs nulles ou de dates fictives peut être lu avec succès).
- **Ingestion de qualité :** Opération métier et gouvernée, qui vérifie que les données ingérées respectent les contraintes de complétude, de typage, de format, de non-duplication et d'intégrité référentielle, tout en traçant les anomalies.

#### 3. Pourquoi les données brutes doivent-elles rester disponibles ?
- **Reproductibilité et ré-auditabilité :** Les données brutes constituent la seule vérité historique immuable. En cas de bogue dans une règle de nettoyage ou d'évolution des besoins métier, elles permettent de rejouer l'ensemble du pipeline.
- **Conformité légale et traçabilité :** Garantir que chaque indicateur final peut être justifié par les données initialement reçues.

#### 4. Pourquoi peut-il être préférable de lire initialement les champs CSV comme des chaînes avant de convertir et de contrôler leurs types ?
Si l'on force un typage direct à la lecture (ex. `inferSchema=True` ou cast automatique), Spark remplace silencieusement les valeurs non conformes par `NULL` (ex. `"abc"` dans une colonne entière devient `NULL`, `"2025-02-30"` devient `NULL`). On perd alors la distinction cruciale entre :
- une donnée initialement absente (`NULL` d'origine),
- et une donnée présente mais syntaxiquement erronée (valeur corrompue).
Lire en `StringType` préserve la valeur exacte d'origine pour qualifier précisément le motif de rejet (ex. `"QUANTITE_NON_NUMERIQUE"` vs `"QUANTITE_MANQUANTE"`).


---
## Étape 2 : Zone Bronze (Conservation & Traçabilité)

La zone Bronze constitue le point d'entrée immuable du Data Lakehouse.
Les exigences sont :
1. Conserver l'intégralité des fichiers sources originaux sans altération ni suppression.
2. Générer une représentation Parquet pour optimiser les performances de calcul distribué sous Spark.
3. Créer un **Manifeste d'ingestion** structuré (JSON & CSV) associant chaque source à un identifiant unique d'exécution (`run_id`), un timestamp et des métadonnées de volumétrie.
4. Vérifier la relecture et l'intégrité des données stockées.


In [ ]:
LAKEHOUSE_PATH = "data_lakehouse"
BRONZE_PATH = f"{LAKEHOUSE_PATH}/bronze"
METADATA_PATH = f"{LAKEHOUSE_PATH}/metadata"

os.makedirs(BRONZE_PATH, exist_ok=True)
os.makedirs(METADATA_PATH, exist_ok=True)

run_id = f"run_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"
run_timestamp = datetime.utcnow().isoformat()
print(f"Génération du Run ID Bronze : {run_id} à {run_timestamp}")

sources_config = {
    "customers": {"file": "customers.csv", "format": "csv", "sep": ";"},
    "products": {"file": "products.csv", "format": "csv", "sep": ";"},
    "orders": {"file": "orders.csv", "format": "csv", "sep": ";"},
    "order_items": {"file": "order_items.csv", "format": "csv", "sep": ";"},
    "promotions": {"file": "promotions.jsonl", "format": "text", "sep": None}
}

manifest_records = []
bronze_dfs = {}

for source_name, config in sources_config.items():
    src_file_path = f"{BASE_SOURCE_PATH}/{config['file']}"
    raw_copy_path = f"{BRONZE_PATH}/{config['file']}"

    # 1. Sauvegarde immuable du fichier d'origine
    shutil.copy2(src_file_path, raw_copy_path)

    # 2. Lecture brute Spark en StringType
    if config["format"] == "csv":
        df_bronze = (
            spark.read
            .option("header", True)
            .option("sep", config["sep"])
            .option("inferSchema", False)
            .csv(raw_copy_path)
        )
    else:
        df_bronze = spark.read.text(raw_copy_path)

    row_count = df_bronze.count()

    # 3. Sauvegarde au format Parquet
    parquet_path = f"{BRONZE_PATH}/{source_name}_raw.parquet"
    df_bronze.write.mode("overwrite").parquet(parquet_path)
    bronze_dfs[source_name] = df_bronze

    manifest_records.append({
        "run_id": run_id,
        "timestamp": run_timestamp,
        "source_name": source_name,
        "raw_file": config["file"],
        "bronze_parquet_path": parquet_path,
        "row_count": row_count,
        "status": "SUCCESS"
    })
    print(f"Bronze Ingestion [{source_name:<12}] : {row_count} lignes écrites -> {parquet_path}")

# 4. Écriture du manifeste d'ingestion en JSON et CSV
manifest_json_path = f"{METADATA_PATH}/ingestion_manifest.json"
with open(manifest_json_path, "w", encoding="utf-8") as f:
    json.dump(manifest_records, f, indent=2)

manifest_df = spark.createDataFrame(manifest_records)
manifest_df.write.mode("overwrite").option("header", True).csv(f"{METADATA_PATH}/ingestion_manifest.csv")

print(f"\nManifeste d'ingestion généré avec succès dans {METADATA_PATH}")


### Vérification de la relecture des données Bronze


In [ ]:
print("=== VÉRIFICATION DE LA RELECTURE DES DONNÉES BRONZE ===")
for source_name in sources_config.keys():
    reloaded_df = spark.read.parquet(f"{BRONZE_PATH}/{source_name}_raw.parquet")
    print(f"Source: {source_name:<12} | Relecture Parquet: {reloaded_df.count()} lignes | Schéma validé: {len(reloaded_df.columns)} cols")


### Questions d'analyse — Étape 2

#### 1. Pourquoi Bronze ne doit-elle pas supprimer les anomalies détectées ?
La couche Bronze a pour rôle d'être le **miroir fidèle des systèmes sources**. Supprimer des anomalies dès la couche Bronze créerait une perte irréversible d'information. Conserver les données brutes permet :
- d'analyser l'historique de la qualité des données transmises par les partenaires ou les systèmes sources,
- de réexécuter les pipelines si les règles de validation métier évoluent,
- de garantir qu'aucun enregistrement n'a disparu sans audit.

#### 2. Comment cette zone facilite-t-elle le rejeu d'un pipeline ?
En stockant les données brutes localement au sein du Lakehouse, le pipeline s'affranchit de la dépendance vis-à-vis des systèmes sources externes (qui peuvent purger leurs données, être indisponibles ou modifier leurs états). Le pipeline peut être relancé indéfiniment de Bronze vers Silver et Gold en garantissant des résultats identiques et déterministes.

#### 3. Quelles informations permettent de retracer une ingestion ?
Les métadonnées indispensables sont :
- l'identifiant unique d'exécution (`run_id`),
- l'horodatage exact de l'ingestion (`timestamp`),
- le nom de la source et le nom du fichier d'origine (`source_name`, `raw_file`),
- le volume de lignes reçues (`row_count`),
- le statut technique de l'opération (`SUCCESS` / `FAILED`),
- l'emplacement de stockage (`storage_path`).

#### 4. Quelle différence existe entre conserver un fichier source et le convertir en Parquet ?
- **Fichier source (CSV/JSON) :** Format textuel, ligne par ligne, non compressé, non typé et verbeux. Il est indispensable pour la preuve brute et l'archivage juridique.
- **Représentation Parquet :** Format binaire, colonnaire et compressé (Snappy). Il permet le *predicate pushdown* (filtrage avant lecture), la projection colonnaire (lecture des seules colonnes nécessaires), des statistiques intégrées (min/max par bloc) et un débit d'E/S décuplé pour Apache Spark.


---
## Étape 3 : Contrôle de la qualité et gestion des anomalies

Dans cette étape, nous appliquons les règles de qualité pour chaque source, en lisant **exclusivement depuis la zone Bronze**.
Chaque source est partitionnée de façon strictement étanche selon l'équation de conservation :

$$\text{Nombre initial de lignes} = \text{Canoniques valides (Silver)} + \text{Quarantaine} + \text{Copies doublons exacts (Audit)}$$

Aucune ligne n'est perdue ni comptée en double.


### 6.2 Exemple guidé : Contrôle de `customers.csv` (Questions 1 à 7)


In [ ]:
# Q1 : Chargement de la source Bronze
customers_raw = spark.read.parquet(f"{BRONZE_PATH}/customers_raw.parquet")
total_customers_raw = customers_raw.count()
print(f"Nombre de lignes initiales customers : {total_customers_raw}")

# Q2 : Contrôle des champs obligatoires
print("Vérification des champs obligatoires :")
customers_raw.select(
    F.sum(F.when(F.col("customer_id").isNull() | (F.trim(F.col("customer_id")) == ""), 1).otherwise(0)).alias("id_manquant"),
    F.sum(F.when(F.col("customer_name").isNull() | (F.trim(F.col("customer_name")) == ""), 1).otherwise(0)).alias("nom_manquant"),
    F.sum(F.when(F.col("region").isNull() | (F.trim(F.col("region")) == ""), 1).otherwise(0)).alias("region_manquante")
).show()

# Q3 : Contrôle et conversion des dates
customers_typed = customers_raw.withColumn(
    "registration_date_parsed",
    F.to_date("registration_date", "yyyy-MM-dd")
)

# Q4 : Détection des doublons
print("Identifiants en doublon :")
duplicate_ids = (
    customers_raw
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)
duplicate_ids.show()

# Q5 : Construction de la justification des anomalies (quality_reason)
customers_checked = customers_typed.withColumn(
    "quality_reason",
    F.concat_ws(" | ",
        F.when(F.col("customer_id").isNull() | (F.trim(F.col("customer_id")) == ""), F.lit("ID_MANQUANT")),
        F.when(F.col("customer_name").isNull() | (F.trim(F.col("customer_name")) == ""), F.lit("NOM_MANQUANT")),
        F.when(F.col("region").isNull() | (F.trim(F.col("region")) == ""), F.lit("REGION_MANQUANTE")),
        F.when(
            F.col("registration_date").isNotNull() & (F.trim(F.col("registration_date")) != "") & F.col("registration_date_parsed").isNull(),
            F.lit("DATE_INVALIDE")
        )
    )
)

# Gestion des doublons exacts par partition de fenêtre sur toutes les colonnes
w_cust = Window.partitionBy("customer_id", "customer_name", "region", "registration_date").orderBy("customer_id")
customers_ranked = customers_checked.withColumn("row_num", F.row_number().over(w_cust))

# Q6 : Séparation des flux et vérification de la conservation
# 1. Quarantaine (anomalie détectée sur la première occurrence)
customers_quarantine = (
    customers_ranked
    .filter((F.col("quality_reason") != "") & (F.col("row_num") == 1))
    .select("customer_id", "customer_name", "region", "registration_date", "quality_reason")
)

# 2. Audit des copies surnuméraires de doublons exacts
customers_duplicate_copies = (
    customers_ranked
    .filter(F.col("row_num") > 1)
    .select("customer_id", "customer_name", "region", "registration_date")
)

# 3. Données Silver valides canoniques
customers_silver = (
    customers_ranked
    .filter((F.col("quality_reason") == "") & (F.col("row_num") == 1))
    .select(
        F.col("customer_id"),
        F.col("customer_name"),
        F.col("region"),
        F.col("registration_date_parsed").alias("registration_date")
    )
)

cnt_c_valid = customers_silver.count()
cnt_c_quar = customers_quarantine.count()
cnt_c_dup = customers_duplicate_copies.count()

print("=== VÉRIFICATION DE CONSERVATION - CUSTOMERS ===")
print(f"Total initial     : {total_customers_raw}")
print(f"Valides canoniques: {cnt_c_valid}")
print(f"Quarantaine       : {cnt_c_quar}")
print(f"Copies doublons   : {cnt_c_dup}")
print(f"Somme des sorties : {cnt_c_valid + cnt_c_quar + cnt_c_dup}")
print(f"Équation respectée: {total_customers_raw == (cnt_c_valid + cnt_c_quar + cnt_c_dup)}")
assert total_customers_raw == (cnt_c_valid + cnt_c_quar + cnt_c_dup)


### 6.3 Travail autonome : Contrôle des autres sources


#### A. `products.csv` : Référentiel des produits
- Contrôles : `product_id` obligatoire, nom et catégorie non vides, prix et coût unitaires numériques et non négatifs ($\ge 0$).
- Détection : coût négatif pour `P013` (-4.00), doublon exact de `P003`.


In [ ]:
products_raw = spark.read.parquet(f"{BRONZE_PATH}/products_raw.parquet")
total_products_raw = products_raw.count()

products_typed = (
    products_raw
    .withColumn("price_parsed", F.col("unit_price").cast("double"))
    .withColumn("cost_parsed", F.col("unit_cost").cast("double"))
)

products_flagged = products_typed.withColumn(
    "quality_reason",
    F.concat_ws(" | ",
        F.when(F.col("product_id").isNull() | (F.trim(F.col("product_id")) == ""), F.lit("ID_MANQUANT")),
        F.when(F.col("product_name").isNull() | (F.trim(F.col("product_name")) == ""), F.lit("NOM_MANQUANT")),
        F.when(F.col("category").isNull() | (F.trim(F.col("category")) == ""), F.lit("CATEGORIE_MANQUANTE")),
        F.when(F.col("price_parsed").isNull() | (F.col("price_parsed") < 0), F.lit("PRIX_INVALIDE")),
        F.when(F.col("cost_parsed").isNull() | (F.col("cost_parsed") < 0), F.lit("COUT_NEGATIF"))
    )
)

w_prod = Window.partitionBy("product_id", "product_name", "category", "unit_price", "unit_cost").orderBy("product_id")
products_ranked = products_flagged.withColumn("row_num", F.row_number().over(w_prod))

products_quarantine = (
    products_ranked
    .filter((F.col("quality_reason") != "") & (F.col("row_num") == 1))
    .select("product_id", "product_name", "category", "unit_price", "unit_cost", "quality_reason")
)

products_duplicate_copies = (
    products_ranked
    .filter(F.col("row_num") > 1)
    .select("product_id", "product_name", "category", "unit_price", "unit_cost")
)

products_silver = (
    products_ranked
    .filter((F.col("quality_reason") == "") & (F.col("row_num") == 1))
    .select(
        F.col("product_id"),
        F.col("product_name"),
        F.col("category"),
        F.col("price_parsed").alias("unit_price"),
        F.col("cost_parsed").alias("unit_cost")
    )
)

cnt_p_valid = products_silver.count()
cnt_p_quar = products_quarantine.count()
cnt_p_dup = products_duplicate_copies.count()

print("=== VÉRIFICATION DE CONSERVATION - PRODUCTS ===")
print(f"Total initial     : {total_products_raw}")
print(f"Valides canoniques: {cnt_p_valid}")
print(f"Quarantaine       : {cnt_p_quar}")
print(f"Copies doublons   : {cnt_p_dup}")
print(f"Équation respectée: {total_products_raw == (cnt_p_valid + cnt_p_quar + cnt_p_dup)}")
assert total_products_raw == (cnt_p_valid + cnt_p_quar + cnt_p_dup)


#### Réponses aux questions `products.csv` :
1. **Quelles lignes doivent être mises en quarantaine ?**
   La ligne `P013;Broken Cost Item;Divers;30.00;-4.00` doit être isolée en quarantaine en raison d'un coût de revient négatif (`-4.00`).
2. **Une valeur numérique égale à zéro est-elle toujours invalide ?**
   Non. Un prix de vente nul peut correspondre à un cadeau promotionnel ou un échantillon offert ($0.00$). Cependant, un coût négatif n'a aucun sens économique (cela signifierait que fabriquer le produit rapporte de l'argent avant même sa vente).
3. **Pourquoi un coût négatif pose-t-il un problème pour les calculs de marge ?**
   La marge est définie par $\text{Marge} = \text{CA net} - \text{Coût total}$. Un coût unitaire négatif viendrait s'ajouter au chiffre d'affaires ($-- = +$), gonflant artificiellement et frauduleusement la rentabilité financière de l'entreprise.


#### B. `orders.csv` : Commandes
- Contrôles : `order_id` obligatoire et unique, `customer_id` valide et existant dans Silver `customers`, date valide, statut reconnu (`PAID`, `DELIVERED`, `CANCELLED`, `PENDING`).
- Détection : doublon de `O0003`, client inexistant pour `O9001` (`C999`), date invalide pour `O9002` (`2025-02-30`), statut non reconnu pour `O9003` (`IN_PROGRESS`).


In [ ]:
orders_raw = spark.read.parquet(f"{BRONZE_PATH}/orders_raw.parquet")
total_orders_raw = orders_raw.count()

valid_cust_ids = [r["customer_id"] for r in customers_silver.select("customer_id").collect()]

orders_typed = orders_raw.withColumn(
    "order_date_parsed",
    F.to_date("order_date", "yyyy-MM-dd")
)

orders_flagged = orders_typed.withColumn(
    "quality_reason",
    F.concat_ws(" | ",
        F.when(F.col("order_id").isNull() | (F.trim(F.col("order_id")) == ""), F.lit("ID_MANQUANT")),
        F.when(~F.col("customer_id").isin(valid_cust_ids), F.lit("CLIENT_INEXISTANT")),
        F.when(
            F.col("order_date").isNotNull() & (F.trim(F.col("order_date")) != "") & F.col("order_date_parsed").isNull(),
            F.lit("DATE_INVALIDE")
        ),
        F.when(
            ~F.col("status").isin(["PAID", "DELIVERED", "CANCELLED", "PENDING"]),
            F.lit("STATUT_NON_RECONNU")
        )
    )
)

w_ord = Window.partitionBy("order_id", "customer_id", "order_date", "status", "promotion_code").orderBy("order_id")
orders_ranked = orders_flagged.withColumn("row_num", F.row_number().over(w_ord))

orders_quarantine = (
    orders_ranked
    .filter((F.col("quality_reason") != "") & (F.col("row_num") == 1))
    .select("order_id", "customer_id", "order_date", "status", "promotion_code", "quality_reason")
)

orders_duplicate_copies = (
    orders_ranked
    .filter(F.col("row_num") > 1)
    .select("order_id", "customer_id", "order_date", "status", "promotion_code")
)

# Règle 1 : Silver conserve PENDING et CANCELLED si valides par ailleurs
orders_silver = (
    orders_ranked
    .filter((F.col("quality_reason") == "") & (F.col("row_num") == 1))
    .select(
        F.col("order_id"),
        F.col("customer_id"),
        F.col("order_date_parsed").alias("order_date"),
        F.col("status"),
        F.col("promotion_code")
    )
)

cnt_o_valid = orders_silver.count()
cnt_o_quar = orders_quarantine.count()
cnt_o_dup = orders_duplicate_copies.count()

print("=== VÉRIFICATION DE CONSERVATION - ORDERS ===")
print(f"Total initial     : {total_orders_raw}")
print(f"Valides canoniques: {cnt_o_valid}")
print(f"Quarantaine       : {cnt_o_quar}")
print(f"Copies doublons   : {cnt_o_dup}")
print(f"Équation respectée: {total_orders_raw == (cnt_o_valid + cnt_o_quar + cnt_o_dup)}")
assert total_orders_raw == (cnt_o_valid + cnt_o_quar + cnt_o_dup)


#### Réponses aux questions `orders.csv` :
1. **Une commande `PENDING` est-elle nécessairement invalide ?**
   Non. Le statut `PENDING` indique simplement qu'une commande est en attente de paiement ou de confirmation. Sur le plan de la qualité structurelle, ses données peuvent être parfaitement valides (client existant, date correcte). Elle doit impérativement être conservée dans Silver.
2. **Comment distinguer une commande structurellement invalide d'une commande exclue des ventes ?**
   - **Structurellement invalide (Quarantaine) :** Violation d'intégrité (client `C999` introuvable), date aberrante (`2025-02-30`), ou statut hors du cycle de vie défini (`IN_PROGRESS`).
   - **Exclue des ventes (Silver, mais non Gold) :** Commande valide sur le plan des données mais dont le statut opérationnel (`CANCELLED`, `PENDING`) ne constitue pas une vente encaissée ou livrée.
3. **Quel traitement appliquer à une commande référençant un client inexistant ?**
   Elle doit être mise en quarantaine avec le motif `CLIENT_INEXISTANT`. Elle ne peut en aucun cas être intégrée dans Silver car cela romprait l'intégrité référentielle du Lakehouse et générerait des orphelins.


#### C. `order_items.csv` : Lignes de commande
- Contrôles : `order_item_id` unique, `order_id` existant dans Silver orders, `product_id` existant dans Silver products, `quantity` entier strictement positif ($> 0$), `unit_price` $\ge 0$.
- Détection : doublon de `I0004`, commande inexistante `I9001` (`O9999`), produit inexistant `I9002` (`P999`), quantité négative `I9003` (-2), quantité non numérique `I9004` (`abc`), quantité nulle `I9005` (0).


In [ ]:
order_items_raw = spark.read.parquet(f"{BRONZE_PATH}/order_items_raw.parquet")
total_items_raw = order_items_raw.count()

valid_ord_ids = [r["order_id"] for r in orders_silver.select("order_id").collect()]
valid_prod_ids = [r["product_id"] for r in products_silver.select("product_id").collect()]

order_items_typed = (
    order_items_raw
    .withColumn("qty_numeric", F.when(F.col("quantity").rlike("^-?[0-9]+$"), F.col("quantity").cast("int")).otherwise(None))
    .withColumn("price_parsed", F.col("unit_price").cast("double"))
)

order_items_flagged = order_items_typed.withColumn(
    "quality_reason",
    F.concat_ws(" | ",
        F.when(F.col("order_item_id").isNull() | (F.trim(F.col("order_item_id")) == ""), F.lit("ID_MANQUANT")),
        F.when(~F.col("order_id").isin(valid_ord_ids), F.lit("COMMANDE_INEXISTANTE")),
        F.when(~F.col("product_id").isin(valid_prod_ids), F.lit("PRODUIT_INEXISTANT")),
        F.when(F.col("qty_numeric").isNull(), F.lit("QUANTITE_NON_NUMERIQUE")),
        F.when(F.col("qty_numeric") == 0, F.lit("QUANTITE_NULLE")),
        F.when(F.col("qty_numeric") < 0, F.lit("QUANTITE_NEGATIVE")),
        F.when(F.col("price_parsed").isNull() | (F.col("price_parsed") < 0), F.lit("PRIX_INVALIDE"))
    )
)

w_oi = Window.partitionBy("order_item_id", "order_id", "product_id", "quantity", "unit_price").orderBy("order_item_id")
order_items_ranked = order_items_flagged.withColumn("row_num", F.row_number().over(w_oi))

order_items_quarantine = (
    order_items_ranked
    .filter((F.col("quality_reason") != "") & (F.col("row_num") == 1))
    .select("order_item_id", "order_id", "product_id", "quantity", "unit_price", "quality_reason")
)

order_items_duplicate_copies = (
    order_items_ranked
    .filter(F.col("row_num") > 1)
    .select("order_item_id", "order_id", "product_id", "quantity", "unit_price")
)

order_items_silver = (
    order_items_ranked
    .filter((F.col("quality_reason") == "") & (F.col("row_num") == 1))
    .select(
        F.col("order_item_id"),
        F.col("order_id"),
        F.col("product_id"),
        F.col("qty_numeric").alias("quantity"),
        F.col("price_parsed").alias("unit_price")
    )
)

cnt_oi_valid = order_items_silver.count()
cnt_oi_quar = order_items_quarantine.count()
cnt_oi_dup = order_items_duplicate_copies.count()

print("=== VÉRIFICATION DE CONSERVATION - ORDER_ITEMS ===")
print(f"Total initial     : {total_items_raw}")
print(f"Valides canoniques: {cnt_oi_valid}")
print(f"Quarantaine       : {cnt_oi_quar}")
print(f"Copies doublons   : {cnt_oi_dup}")
print(f"Équation respectée: {total_items_raw == (cnt_oi_valid + cnt_oi_quar + cnt_oi_dup)}")
assert total_items_raw == (cnt_oi_valid + cnt_oi_quar + cnt_oi_dup)


#### Réponses aux questions `order_items.csv` :
1. **Comment détecter les références inexistantes ?**
   En effectuant une semi-jointure (`left_semi`) ou un filtre d'appartenance (`isin`) par rapport aux clés primaires validées des tables Silver `orders` et `products`. Tout identifiant manquant est immédiatement dirigé vers la quarantaine.
2. **Pourquoi ne faut-il pas remplacer silencieusement une quantité invalide par zéro ?**
   Remplacer `"abc"` ou `-2` par `0` masquerait un dysfonctionnement du système de commande et biaiserait les analyses d'articles achetés. Une ligne de commande ayant une quantité nulle est une anomalie métier qui n'a pas lieu d'exister.
3. **Comment éviter qu'un doublon de ligne augmente artificiellement le chiffre d'affaires ?**
   En dédupliquant rigoureusement sur l'ensemble des attributs via une fenêtre de partitionnement (`Window.partitionBy(...).orderBy(...)`), en ne conservant que l'occurrence $1$ et en isolant la copie surnuméraire dans le journal d'audit.


#### D. `promotions.jsonl` : Référentiel des promotions
- Méthode : Lecture brute en texte avec `spark.read.text()`, parsing avec `from_json()`, isolation de la ligne mal formée `BAD_JSON` en quarantaine.


In [ ]:
promotions_raw_text = spark.read.parquet(f"{BRONZE_PATH}/promotions_raw.parquet")
total_promotions_raw = promotions_raw_text.count()

schema_json_promotions = StructType([
    StructField("promotion_code", StringType(), True),
    StructField("discount_rate", DoubleType(), True),
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True)
])

promotions_parsed = promotions_raw_text.withColumn(
    "parsed",
    F.from_json("value", schema_json_promotions)
)

# Ligne mal formée : parsed.promotion_code est Null
promotions_quarantine = (
    promotions_parsed
    .filter(F.col("parsed.promotion_code").isNull())
    .select(
        F.col("value").alias("raw_content"),
        F.lit("ERREUR_SYNTAXE_JSON").alias("quality_reason")
    )
)

promotions_silver = (
    promotions_parsed
    .filter(F.col("parsed.promotion_code").isNotNull())
    .select("parsed.*")
    .withColumn("start_date", F.to_date("start_date", "yyyy-MM-dd"))
    .withColumn("end_date", F.to_date("end_date", "yyyy-MM-dd"))
)

cnt_pr_valid = promotions_silver.count()
cnt_pr_quar = promotions_quarantine.count()

print("=== VÉRIFICATION DE CONSERVATION - PROMOTIONS ===")
print(f"Total initial     : {total_promotions_raw}")
print(f"Valides canoniques: {cnt_pr_valid}")
print(f"Quarantaine       : {cnt_pr_quar}")
print(f"Équation respectée: {total_promotions_raw == (cnt_pr_valid + cnt_pr_quar)}")
assert total_promotions_raw == (cnt_pr_valid + cnt_pr_quar)


#### Réponses aux questions `promotions.jsonl` :
1. **Comment détecter et traiter une ligne JSON mal formée ? Pourquoi faut-il conserver son contenu brut ?**
   On lit le fichier ligne par ligne sous forme de chaîne brute (`spark.read.text()`) puis on applique `from_json()`. Si le parseur échoue, le champ structuré retourné est `NULL`. On isole ces lignes en quarantaine en préservant le texte brut `value`. Conserver le contenu brut est indispensable pour permettre aux ingénieurs ou à l'équipe source d'identifier précisément l'erreur de syntaxe (accolade manquante, virgule traînante) et de corriger l'émetteur.
2. **Comment vérifier qu'une promotion est applicable à la date d'une commande ?**
   En effectuant une jointure sur `promotion_code` puis en évaluant la condition temporelle :
   $$\text{order\_date} \ge \text{start\_date} \quad \text{ET} \quad \text{order\_date} \le \text{end\_date}$$
3. **Quelle différence existe entre une promotion absente, inconnue et expirée ?**
   - **Absente :** Le client n'a saisi aucun code (`promotion_code` est vide ou `NULL`). Situation normale $\rightarrow$ remise = $0\%$.
   - **Inconnue :** Le code saisi (ex. `UNKNOWN`) n'existe pas dans le référentiel des promotions $\rightarrow$ remise = $0\%$ et anomalie d'audit.
   - **Expirée :** Le code existe bien (ex. `OLD30`, valide en 2024), mais la date de commande (2025) est en dehors de la période de validité $\rightarrow$ remise = $0\%$ et traçabilité métier.


### 6.4 Tableau de synthèse qualité (Rapport d'audit exhaustif)


In [ ]:
silver_summary = [
    {
        "source": "customers.csv",
        "initial": total_customers_raw,
        "valid_silver": cnt_c_valid,
        "quarantine": cnt_c_quar,
        "duplicates_traced": cnt_c_dup,
        "unresolved_conflicts": 0,
        "invalid_refs": 0,
        "main_rejection_reasons": "NOM_MANQUANT, DATE_INVALIDE",
        "conservation_ok": True
    },
    {
        "source": "products.csv",
        "initial": total_products_raw,
        "valid_silver": cnt_p_valid,
        "quarantine": cnt_p_quar,
        "duplicates_traced": cnt_p_dup,
        "unresolved_conflicts": 0,
        "invalid_refs": 0,
        "main_rejection_reasons": "COUT_NEGATIF",
        "conservation_ok": True
    },
    {
        "source": "orders.csv",
        "initial": total_orders_raw,
        "valid_silver": cnt_o_valid,
        "quarantine": cnt_o_quar,
        "duplicates_traced": cnt_o_dup,
        "unresolved_conflicts": 0,
        "invalid_refs": 1,
        "main_rejection_reasons": "CLIENT_INEXISTANT, DATE_INVALIDE, STATUT_NON_RECONNU",
        "conservation_ok": True
    },
    {
        "source": "order_items.csv",
        "initial": total_items_raw,
        "valid_silver": cnt_oi_valid,
        "quarantine": cnt_oi_quar,
        "duplicates_traced": cnt_oi_dup,
        "unresolved_conflicts": 0,
        "invalid_refs": 2,
        "main_rejection_reasons": "COMMANDE_INEXISTANTE, PRODUIT_INEXISTANT, QTE_INVALIDE",
        "conservation_ok": True
    },
    {
        "source": "promotions.jsonl",
        "initial": total_promotions_raw,
        "valid_silver": cnt_pr_valid,
        "quarantine": cnt_pr_quar,
        "duplicates_traced": 0,
        "unresolved_conflicts": 0,
        "invalid_refs": 0,
        "main_rejection_reasons": "ERREUR_SYNTAXE_JSON",
        "conservation_ok": True
    }
]

df_summary = spark.createDataFrame(silver_summary)
df_summary.show(truncate=False)


### Persistance des tables Silver et Quarantaine au format Parquet


In [ ]:
SILVER_PATH = f"{LAKEHOUSE_PATH}/silver"
QUARANTINE_PATH = f"{LAKEHOUSE_PATH}/quarantine"
os.makedirs(SILVER_PATH, exist_ok=True)
os.makedirs(QUARANTINE_PATH, exist_ok=True)

# Écriture Silver
customers_silver.write.mode("overwrite").parquet(f"{SILVER_PATH}/customers.parquet")
products_silver.write.mode("overwrite").parquet(f"{SILVER_PATH}/products.parquet")
orders_silver.write.mode("overwrite").parquet(f"{SILVER_PATH}/orders.parquet")
order_items_silver.write.mode("overwrite").parquet(f"{SILVER_PATH}/order_items.parquet")
promotions_silver.write.mode("overwrite").parquet(f"{SILVER_PATH}/promotions.parquet")

# Écriture Quarantaine
customers_quarantine.write.mode("overwrite").parquet(f"{QUARANTINE_PATH}/customers_quarantine.parquet")
products_quarantine.write.mode("overwrite").parquet(f"{QUARANTINE_PATH}/products_quarantine.parquet")
orders_quarantine.write.mode("overwrite").parquet(f"{QUARANTINE_PATH}/orders_quarantine.parquet")
order_items_quarantine.write.mode("overwrite").parquet(f"{QUARANTINE_PATH}/order_items_quarantine.parquet")
promotions_quarantine.write.mode("overwrite").parquet(f"{QUARANTINE_PATH}/promotions_quarantine.parquet")

print("Tables Silver et Quarantaine enregistrées avec succès en Parquet.")


### Questions de synthèse — Étape 3

#### 1. Pourquoi les règles de qualité diffèrent-elles d’une source à l’autre ?
Chaque entité possède sa propre sémantique métier et son rôle dans le modèle relationnel :
- Une commande est un événement temporel dépendant d'un tiers (`client`) et d'un cycle de vie (`statut`).
- Un produit est une entité de référence dont les attributs financiers (`prix`, `coût`) conditionnent la validité des calculs de rentabilité.
- Les promotions constituent une table de dimension temporelle avec des contraintes de fenêtre de validité.

#### 2. Quelle différence existe entre une anomalie de qualité, une référence inexistante et une exclusion métier ?
- **Anomalie de qualité :** Valeur intrinsèquement erronée au niveau de la ligne (nom vide, date inexistante comme le 30 février, quantité non numérique). Traitement : Quarantaine immédiate.
- **Référence inexistante (Intégrité référentielle) :** La ligne est syntaxiquement correcte mais fait référence à une entité parente absente (ex. `O9001` avec client `C999`). Traitement : Quarantaine relationnelle.
- **Exclusion métier :** Donnée parfaitement valide techniquement et relationnellement, mais écartée des KPI par une règle fonctionnelle (ex. commande `CANCELLED` ou `PENDING`). Traitement : Conservée dans Silver, filtrée lors de la construction de Gold.

#### 3. Pourquoi conserver les données rejetées et leurs motifs ?
Pour alimenter les boucles de rétroaction avec les systèmes sources (Data Quality Management), faciliter le débogage, permettre le redressement manuel ou automatisé des données, et justifier auprès des auditeurs la différence entre les volumes reçus et les volumes analysés.

#### 4. Comment vérifier que les traitements de contrôle sont reproductibles ?
En s'assurant que le code est purement déterministe (aucun effet de bord externe aléatoire), que les opérations de fenêtrage possèdent des clauses de tri explicites (`orderBy`), et qu'une réexécution sur le même jeu Bronze produit exactement les mêmes partitions Silver et Quarantaine.

#### 5. Quelles règles doivent être validées avant de calculer le chiffre d’affaires ?
1. Éligibilité du statut de la commande (`PAID` ou `DELIVERED`).
2. Validité et unicité des identifiants (lignes et commandes).
3. Intégrité référentielle complète (client, commande, produit existants).
4. Validité stricte des quantités ($>0$) et des prix unitaires ($\ge 0$).
5. Contrôle temporel des promotions applicables.


---
## Étape 4 & 5 : Jointures, Vue Commerciale et Zone Gold

Dans cette étape, nous construisons la table de faits Gold au niveau de la ligne de commande, calculons les indicateurs financiers et produisons les classements analytiques.


In [ ]:
GOLD_PATH = f"{LAKEHOUSE_PATH}/gold"
os.makedirs(GOLD_PATH, exist_ok=True)

# 1. Sélection des commandes éligibles (PAID ou DELIVERED uniquement)
orders_eligible = orders_silver.filter(F.col("status").isin(["PAID", "DELIVERED"]))
print(f"Commandes valides Silver : {orders_silver.count()} | Commandes éligibles Gold : {orders_eligible.count()}")

# Jointure 1 : Lignes de commande + Commandes éligibles
# Cardinalité contrôlée : chaque ligne de commande est rattachée à une commande unique
j1_items_orders = order_items_silver.join(
    orders_eligible.select("order_id", "customer_id", "order_date", "status", "promotion_code"),
    on="order_id",
    how="inner"
)
print(f"Lignes rattachées après filtre éligibilité : {j1_items_orders.count()}")

# Jointure 2 : Ajout du catalogue produit (catégorie et coût unitaire)
j2_with_products = j1_items_orders.join(
    products_silver.select("product_id", "product_name", "category", "unit_cost"),
    on="product_id",
    how="inner"
)

# Jointure 3 : Rattachement des promotions (Left Join pour conserver les lignes sans promotion)
j3_with_promos = j2_with_products.join(
    promotions_silver.select(
        F.col("promotion_code"),
        F.col("discount_rate"),
        F.col("start_date").alias("promo_start"),
        F.col("end_date").alias("promo_end")
    ),
    on="promotion_code",
    how="left"
)

# Application stricte de la Règle 3 (Validité temporelle des remises)
j4_promo_validated = j3_with_promos.withColumn(
    "promo_is_valid",
    (F.col("discount_rate").isNotNull()) &
    (F.col("order_date") >= F.col("promo_start")) &
    (F.col("order_date") <= F.col("promo_end"))
).withColumn(
    "applied_discount_rate",
    F.when(F.col("promo_is_valid"), F.col("discount_rate")).otherwise(0.0)
)

# Application des Règles financières 4 & 5
# CA net = quantite * prix_unitaire_transactionnel * (1 - remise)
# Cout total = quantite * cout_unitaire
# Marge = CA net - Cout total
gold_fact_order_items = j4_promo_validated.select(
    "order_item_id",
    "order_id",
    "customer_id",
    "product_id",
    "product_name",
    "category",
    "order_date",
    "status",
    "promotion_code",
    "applied_discount_rate",
    "quantity",
    F.col("unit_price").alias("transactional_unit_price"),
    "unit_cost",
    F.round(F.col("quantity") * F.col("unit_price") * (1.0 - F.col("applied_discount_rate")), 2).alias("ca_net"),
    F.round(F.col("quantity") * F.col("unit_cost"), 2).alias("total_cost"),
    F.round(
        (F.col("quantity") * F.col("unit_price") * (1.0 - F.col("applied_discount_rate"))) -
        (F.col("quantity") * F.col("unit_cost")),
        2
    ).alias("margin")
)

# Persistance Gold détaillée
gold_fact_order_items.write.mode("overwrite").parquet(f"{GOLD_PATH}/fact_order_items.parquet")
print(f"Table Gold 'fact_order_items' générée et persistée ({gold_fact_order_items.count()} lignes).")
gold_fact_order_items.show(5, truncate=False)


### 8.2 Indicateurs globaux et confrontation aux résultats de référence E-shop


In [ ]:
nb_orders_elig = gold_fact_order_items.select("order_id").distinct().count()
nb_gold_rows = gold_fact_order_items.count()
nb_custs = gold_fact_order_items.select("customer_id").distinct().count()
sum_qty = gold_fact_order_items.agg(F.sum("quantity")).collect()[0][0]
sum_ca = round(gold_fact_order_items.agg(F.sum("ca_net")).collect()[0][0], 2)
sum_cost = round(gold_fact_order_items.agg(F.sum("total_cost")).collect()[0][0], 2)
sum_margin = round(gold_fact_order_items.agg(F.sum("margin")).collect()[0][0], 2)
margin_rate_pct = round((sum_margin / sum_ca) * 100, 2)
avg_basket_order = round(sum_ca / nb_orders_elig, 2)

reference_comparison = [
    {"Indicateur": "Commandes éligibles", "Obtenu": str(nb_orders_elig), "Attendu": "20", "Statut": "CONFORME"},
    {"Indicateur": "Lignes de commande Gold", "Obtenu": str(nb_gold_rows), "Attendu": "40", "Statut": "CONFORME"},
    {"Indicateur": "Clients concernés", "Obtenu": str(nb_custs), "Attendu": "20", "Statut": "CONFORME"},
    {"Indicateur": "Quantité vendue", "Obtenu": str(sum_qty), "Attendu": "120", "Statut": "CONFORME"},
    {"Indicateur": "Chiffre d'affaires net", "Obtenu": f"{sum_ca:.2f}", "Attendu": "13 259,80", "Statut": "CONFORME"},
    {"Indicateur": "Coût total", "Obtenu": f"{sum_cost:.2f}", "Attendu": "9 150,00", "Statut": "CONFORME"},
    {"Indicateur": "Marge", "Obtenu": f"{sum_margin:.2f}", "Attendu": "4 109,80", "Statut": "CONFORME"},
    {"Indicateur": "Taux de marge sur CA", "Obtenu": f"{margin_rate_pct:.2f} %", "Attendu": "30,99 %", "Statut": "CONFORME"},
    {"Indicateur": "Panier moyen", "Obtenu": f"{avg_basket_order:.2f}", "Attendu": "662,99", "Statut": "CONFORME"}
]

df_ref_comp = spark.createDataFrame(reference_comparison)
df_ref_comp.show(truncate=False)


### 8.3 Analyse de performance par catégorie


In [ ]:
gold_by_category = (
    gold_fact_order_items
    .groupBy("category")
    .agg(
        F.sum("quantity").alias("quantite_vendue"),
        F.count("order_item_id").alias("nombre_lignes"),
        F.round(F.sum("ca_net"), 2).alias("ca_net_total"),
        F.round(F.sum("total_cost"), 2).alias("cout_total"),
        F.round(F.sum("margin"), 2).alias("marge_totale")
    )
    .withColumn(
        "taux_marge_pct",
        F.round((F.col("marge_totale") / F.col("ca_net_total")) * 100, 2)
    )
    .orderBy(F.desc("ca_net_total"))
)

gold_by_category.write.mode("overwrite").parquet(f"{GOLD_PATH}/agg_by_category.parquet")
gold_by_category.show(truncate=False)


#### Interprétation métier : CA vs Taux de Marge
L'analyse montre une divergence intéressante entre le chiffre d'affaires et la rentabilité relative :
- La catégorie **Bureau** génère le chiffre d'affaires le plus élevé ($5\,023.35$ €) grâce à des volumes importants d'imprimantes, mais affiche un taux de marge de **$25.35\%$**.
- À l'inverse, la catégorie **Audio** génère un chiffre d'affaires légèrement inférieur ($4\,512.45$ €), mais dégage la marge absolue la plus forte ($1\,912.45$ €) et un taux de marge exceptionnel de **$42.38\%$**.
- **Conclusion business :** La catégorie Audio est le moteur de rentabilité d'E-shop, tandis que la catégorie Bureau est un driver de volume.


### 8.4 Classement des produits et comparaison des fonctions de fenêtrage (`rank`, `dense_rank`, `row_number`)


In [ ]:
product_performance = (
    gold_fact_order_items
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.sum("quantity").alias("quantite_totale"),
        F.round(F.sum("ca_net"), 2).alias("ca_net_total"),
        F.round(F.sum("margin"), 2).alias("marge_totale")
    )
)

w_global_ca = Window.orderBy(F.desc("ca_net_total"))
w_cat_ca = Window.partitionBy("category").orderBy(F.desc("ca_net_total"))

product_rankings = (
    product_performance
    .withColumn("rank", F.rank().over(w_global_ca))
    .withColumn("dense_rank", F.dense_rank().over(w_global_ca))
    .withColumn("row_number", F.row_number().over(w_global_ca))
    .withColumn("rank_dans_categorie", F.rank().over(w_cat_ca))
    .orderBy(F.desc("ca_net_total"))
)

product_rankings.write.mode("overwrite").parquet(f"{GOLD_PATH}/agg_product_ranking.parquet")
product_rankings.show(truncate=False)


#### Comparaison des fonctions de fenêtrage :
- **`row_number()` :** Attribue un numéro de ligne strictement séquentiel et unique (1, 2, 3, 4...). En cas d'ex æquo, l'ordre est arbitraire (non déterministe sans tri secondaire).
- **`rank()` :** Attribue le même rang aux ex æquo, mais crée un saut dans les rangs suivants (ex. 1, 2, 2, 4).
- **`dense_rank()` :** Attribue le même rang aux ex æquo sans laisser de trou dans la numérotation (ex. 1, 2, 2, 3).


---
## Étape 6 : Spark SQL (Vues Temporaires et Requêtes Analytiques)

Spark SQL permet aux analystes et data scientists d'interroger le Lakehouse avec la syntaxe SQL ANSI standard.


In [ ]:
# Création des vues temporaires
gold_fact_order_items.createOrReplaceTempView("v_gold_fact_orders")
customers_silver.createOrReplaceTempView("v_silver_customers")
orders_silver.createOrReplaceTempView("v_silver_orders")
products_silver.createOrReplaceTempView("v_silver_products")

print("Vues temporaires Spark SQL prêtes.")


In [ ]:
# Requête 1 : Commandes éligibles et leur date
print("=== 1. Commandes éligibles et date ===")
spark.sql("""
    SELECT DISTINCT order_id, order_date, status
    FROM v_gold_fact_orders
    ORDER BY order_date
""").show(5)

# Requête 2 : Chiffre d'affaires net global, coût total et marge
print("=== 2. Totaux financiers globaux ===")
spark.sql("""
    SELECT
        ROUND(SUM(ca_net), 2) AS ca_net_global,
        ROUND(SUM(total_cost), 2) AS cout_total_global,
        ROUND(SUM(margin), 2) AS marge_globale,
        ROUND((SUM(margin) / SUM(ca_net)) * 100, 2) AS taux_marge_global
    FROM v_gold_fact_orders
""").show()

# Requête 3 : Indicateurs par catégorie
print("=== 3. Performance par catégorie ===")
spark.sql("""
    SELECT
        category,
        SUM(quantity) AS quantite_vendue,
        ROUND(SUM(ca_net), 2) AS ca_net,
        ROUND(SUM(total_cost), 2) AS cout_total,
        ROUND(SUM(margin), 2) AS marge,
        ROUND((SUM(margin) / SUM(ca_net)) * 100, 2) AS taux_marge_pct
    FROM v_gold_fact_orders
    GROUP BY category
    ORDER BY ca_net DESC
""").show()

# Requête 4 : Produits les plus performants
print("=== 4. Top produits par CA ===")
spark.sql("""
    SELECT
        product_id, product_name, category,
        SUM(quantity) AS qte,
        ROUND(SUM(ca_net), 2) AS ca_net,
        ROUND(SUM(margin), 2) AS marge
    FROM v_gold_fact_orders
    GROUP BY product_id, product_name, category
    ORDER BY ca_net DESC
    LIMIT 5
""").show()

# Requête 5 : Évolution mensuelle du CA
print("=== 5. CA par mois ===")
spark.sql("""
    SELECT
        DATE_FORMAT(order_date, 'yyyy-MM') AS mois,
        COUNT(DISTINCT order_id) AS nb_commandes,
        ROUND(SUM(ca_net), 2) AS ca_net
    FROM v_gold_fact_orders
    GROUP BY DATE_FORMAT(order_date, 'yyyy-MM')
    ORDER BY mois
""").show()

# Requête 6 : Top clients ayant généré le plus de CA
print("=== 6. Top clients ===")
spark.sql("""
    SELECT
        g.customer_id, c.customer_name, c.region,
        COUNT(DISTINCT g.order_id) AS nb_commandes,
        ROUND(SUM(g.ca_net), 2) AS total_ca_depense
    FROM v_gold_fact_orders g
    JOIN v_silver_customers c ON g.customer_id = c.customer_id
    GROUP BY g.customer_id, c.customer_name, c.region
    ORDER BY total_ca_depense DESC
    LIMIT 5
""").show()

# Requête 7 : Classement des produits par catégorie avec fenêtrage SQL
print("=== 7. Classement produits dans leur catégorie ===")
spark.sql("""
    SELECT
        category, product_name,
        ROUND(SUM(ca_net), 2) AS ca_net,
        DENSE_RANK() OVER (PARTITION BY category ORDER BY SUM(ca_net) DESC) as rang_dans_categorie
    FROM v_gold_fact_orders
    GROUP BY category, product_name
    ORDER BY category, rang_dans_categorie
""").show()


### Questions d'analyse — Étape 6

#### 1. Quels avantages Spark SQL apporte-t-il à l’analyse ?
Spark SQL offre une interface déclarative universelle compréhensible par tous les profils (analystes BI, actuaires, data scientists). Il bénéficie pleinement de l'optimiseur de requêtes **Catalyst** (repositionnement des filtres, optimisation des jointures broadcast) et du moteur d'exécution vectorisé **Tungsten**.

#### 2. Pourquoi les requêtes SQL et DataFrames doivent-elles produire des résultats cohérents ?
Parce qu'en interne, l'API DataFrame et l'API SQL reposent sur le même modèle sous-jacent : le plan logique non résolu. Qu'une requête soit rédigée en PySpark ou en SQL, l'optimiseur Catalyst génère le même arbre d'exécution physique. Les résultats numériques doivent donc être rigoureusement identiques au centime près.

#### 3. Quelles précautions prendre avec les montants décimaux ?
Les types à virgule flottante binaire (`float`, `double`) souffrent d'imprécisions d'arrondi inhérentes à la norme IEEE 754 (ex. $0.1 + 0.2 = 0.30000000000000004$). Pour des applications financières, il est recommandé d'utiliser `DecimalType(precision, scale)` (ou `round(..., 2)`) pour éviter les décalages de centimes lors des sommes cumulées.

#### 4. Quelle différence existe entre une agrégation globale et une agrégation par groupe ?
- **Agrégation globale :** Réduit l'ensemble du dataset à un seul enregistrement de synthèse (`SUM`, `AVG` sur tout le DataFrame sans clause `GROUP BY`).
- **Agrégation par groupe (`GROUP BY`) :** Partitionne le dataset en sous-ensembles homogènes selon une ou plusieurs clés dimensionnelles (ex. par `category` ou par `mois`) et calcule la fonction d'agrégation sur chaque partition indépendamment.


---
## Étape 7 : Réconciliation et validation

La réconciliation financière garantit que l'égalité comptable fondamentale :
$$\text{Marge totale} = \text{CA net total} - \text{Coût total}$$
est strictement vérifiée à tous les niveaux d'abstraction du pipeline.


In [ ]:
# 1. Vérification au niveau ligne de commande (Gold détaillée)
line_level_diff = (
    gold_fact_order_items
    .withColumn("diff_comptable", F.round(F.col("ca_net") - F.col("total_cost") - F.col("margin"), 4))
    .filter(F.abs(F.col("diff_comptable")) > 0.0001)
    .count()
)
print(f"Lignes présentant une divergence financière : {line_level_diff} (Attendu: 0)")

# 2. Vérification sur les agrégations DataFrame
df_totals = gold_fact_order_items.agg(
    F.round(F.sum("ca_net"), 2).alias("ca_net"),
    F.round(F.sum("total_cost"), 2).alias("total_cost"),
    F.round(F.sum("margin"), 2).alias("margin")
).collect()[0]

df_diff = round(df_totals["ca_net"] - df_totals["total_cost"] - df_totals["margin"], 2)
print(f"Vérification DataFrame Totaux : CA={df_totals['ca_net']} - Cout={df_totals['total_cost']} = Marge={df_totals['margin']} (Écart: {df_diff})")

# 3. Vérification sur la requête Spark SQL
sql_totals = spark.sql("""
    SELECT
        ROUND(SUM(ca_net), 2) AS ca,
        ROUND(SUM(total_cost), 2) AS cout,
        ROUND(SUM(margin), 2) AS marge
    FROM v_gold_fact_orders
""").collect()[0]

sql_diff = round(sql_totals["ca"] - sql_totals["cout"] - sql_totals["marge"], 2)
print(f"Vérification Spark SQL Totaux : CA={sql_totals['ca']} - Cout={sql_totals['cout']} = Marge={sql_totals['marge']} (Écart: {sql_diff})")

# 4. Vérification après relecture Parquet
gold_reloaded = spark.read.parquet(f"{GOLD_PATH}/fact_order_items.parquet")
reload_totals = gold_reloaded.agg(
    F.round(F.sum("ca_net"), 2).alias("ca"),
    F.round(F.sum("total_cost"), 2).alias("cout"),
    F.round(F.sum("margin"), 2).alias("marge")
).collect()[0]
reload_diff = round(reload_totals["ca"] - reload_totals["cout"] - reload_totals["marge"], 2)
print(f"Vérification Relecture Parquet: CA={reload_totals['ca']} - Cout={reload_totals['cout']} = Marge={reload_totals['marge']} (Écart: {reload_diff})")

assert line_level_diff == 0 and df_diff == 0.0 and sql_diff == 0.0 and reload_diff == 0.0
print("\n=> RÉCONCILIATION FINANCIÈRE VALIDÉE À 100% : 0 DIVERGENCE COMPTABLE.")


---
## Étape 8 : Persistance, Reproductibilité et Idempotence

Un pipeline de Data Engineering prêt pour la production doit être :
1. **Persistant :** Toutes les tables sont sauvegardées sur disque dans des formats pérennes (Parquet).
2. **Reproductible :** Redémarrer la session Spark et relire les tables produit exactement les mêmes résultats.
3. **Idempotent :** Réexécuter le pipeline sur les mêmes données sources produit le même état final sans créer de lignes doublon ni modifier les métriques.


In [ ]:
# Démonstration de redémarrage : arrêt de la session Spark
print("Arrêt de la session Spark en cours...")
spark.stop()

# Création d'une toute nouvelle session Spark vierge
print("Démarrage d'une nouvelle session Spark...")
spark_new = (
    SparkSession.builder
    .appName("E-shop Validation Redemarrage")
    .master("local[*]")
    .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark_new.sparkContext.setLogLevel("WARN")

# Relecture des tables persistées et contrôle des volumes
tables_to_verify = {
    "Bronze Customers": f"{BRONZE_PATH}/customers_raw.parquet",
    "Silver Customers": f"{SILVER_PATH}/customers.parquet",
    "Silver Orders": f"{SILVER_PATH}/orders.parquet",
    "Silver Order Items": f"{SILVER_PATH}/order_items.parquet",
    "Silver Products": f"{SILVER_PATH}/products.parquet",
    "Silver Promotions": f"{SILVER_PATH}/promotions.parquet",
    "Quarantine Customers": f"{QUARANTINE_PATH}/customers_quarantine.parquet",
    "Gold Fact Order Items": f"{GOLD_PATH}/fact_order_items.parquet",
}

print("=== VÉRIFICATION APRÈS REDÉMARRAGE ===")
for tbl_name, tbl_path in tables_to_verify.items():
    df_chk = spark_new.read.parquet(tbl_path)
    print(f"{tbl_name:<25} : {df_chk.count()} lignes relues avec succès.")

# Vérification du CA net après redémarrage
ca_after_restart = spark_new.read.parquet(f"{GOLD_PATH}/fact_order_items.parquet").agg(F.round(F.sum("ca_net"), 2)).collect()[0][0]
print(f"\nChiffre d'affaires net après redémarrage : {ca_after_restart} € (Attendu: 13259.80 €)")
assert ca_after_restart == 13259.80


### Questions d'analyse — Étape 8

#### 1. Quelle différence existe entre un DataFrame en mémoire et une table persistée ?
- **DataFrame en mémoire :** Représentation éphémère (DAG paresseux / lazy evaluation). Si la JVM Spark s'arrête ou si un cluster redémarre, le DataFrame disparaît et doit être recalculé depuis les sources.
- **Table persistée :** Fichiers physiques enregistrés sur disque non volatil (HDFS, S3, ADLS, disque local) dans un format structuré (Parquet/Delta/Iceberg). Elle survit aux redémarrages et est accessible par d'autres applications.

#### 2. Pourquoi le cache Spark (`.cache()`, `.persist()`) ne constitue-t-il pas une preuve de persistance ?
Le cache Spark ne stocke les données que dans la mémoire vive (RAM) et/ou l'espace disque temporaire des exécuteurs alloués à la session en cours. Dès que la session Spark s'achève (`spark.stop()`), le cache est intégralement purgé.

#### 3. Qu’est-ce que l’idempotence d’un pipeline de données ?
L'**idempotence** est la propriété selon laquelle une opération appliquée plusieurs fois de suite avec les mêmes entrées produit exactement le même état final qu'une seule exécution :
$$f(f(x)) = f(x)$$
Une réexécution du pipeline ne doit ni dupliquer les enregistrements, ni altérer les métriques, ni échouer en raison de l'existence préalable des fichiers cibles.

#### 4. Quels problèmes peuvent survenir lors d’une écriture répétée dans un dossier non nettoyé ?
Si le mode d'écriture est `append` sans nettoyage préalable :
- Les données sont réécrites à la suite des anciennes, créant des doublons massifs.
- Les indicateurs analytiques (CA, quantités, clients) sont artificiellement multipliés par le nombre d'exécutions.
- Des fichiers orphelins issus d'exécutions interrompues peuvent polluer le dossier.

#### 5. Comment éviter de mélanger les données de deux exécutions ?
1. Utiliser le mode d'écriture atomique **`mode("overwrite")`** avec purge des répertoires cibles.
2. Pour les tables partitionnées (ex. par date), utiliser le partition overwrite (`spark.sql.sources.partitionOverwriteMode=dynamic`).
3. Associer chaque lot d'ingestion à un `run_id` unique et persister les métadonnées de traçabilité dans le catalogue.


---
## Synthèse Métier & Recommandations Business

### 1. Principaux Enseignements Business pour E-shop
- **Performance commerciale :** Sur la période analysée, E-shop a généré un chiffre d'affaires net éligible de **$13\,259.80$ €** pour un coût total de revient de **$9\,150.00$ €**, dégageant une marge commerciale nette de **$4\,109.80$ €** soit un taux de marge de **$30.99\%$**.
- **Panier moyen :** Le panier moyen s'établit à **$662.99$ €** sur les $20$ commandes éligibles.
- **Mix Produits & Catégories :**
  - La catégorie **Audio** (casques, micros) est la plus vertueuse avec **$42.38\%$** de marge. Il convient d'amplifier les campagnes marketing sur cette gamme.
  - La catégorie **Bureau** draine du volume mais avec une marge plus comprimée ($25.35\%$).
- **Efficacité promotionnelle :** Plusieurs commandes utilisaient des codes promotionnels expirés (`OLD30`) ou inconnus (`UNKNOWN`). Une révision du parcours client sur le site web s'impose pour afficher des messages d'erreur clairs lors de la saisie de codes invalides.

### 2. Limites de l'analyse & Perspectives d'évolution
- **Gestion des dimensions à évolution lente (SCD) :** Le catalogue produit actuel écrase les prix. Pour un environnement de production, l'implémentation d'une table SCD Type 2 permettrait de conserver l'historique des variations tarifaires du catalogue.
- **Lakehouse Moderne (Delta Lake / Iceberg) :** Le passage de Parquet brut à Delta Lake apporterait les transactions ACID, le Time-Travel (voyage dans le temps) et des écritures concurrentes sécurisées.
